# Phase 2 — Data Loading & Cleaning
**UCI Hydraulic Systems Dataset — Predictive Maintenance**

Steps covered in this notebook:
1. Load all 17 sensor `.txt` files and `profile.txt`
2. Filter unstable cycles
3. Encode the 4 target labels as ordinal integers
4. Validate data integrity
5. Save clean data to `data/processed/`
6. Diagnostic plots

## 2.0 — Imports & configuration

In [ ]:
import os
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# ── Paths ──────────────────────────────────────────────────────────
RAW_DIR       = "data/raw"
PROCESSED_DIR = "data/processed"
SENSORS_DIR   = os.path.join(PROCESSED_DIR, "sensors")

os.makedirs(SENSORS_DIR, exist_ok=True)

# ── Sensor metadata ────────────────────────────────────────────────
# Each sensor records for 60 seconds → columns per file = hz × 60
SENSOR_META = {
    "PS1":  {"unit": "bar",   "hz": 100, "type": "pressure"},
    "PS2":  {"unit": "bar",   "hz": 100, "type": "pressure"},
    "PS3":  {"unit": "bar",   "hz": 100, "type": "pressure"},
    "PS4":  {"unit": "bar",   "hz": 100, "type": "pressure"},
    "PS5":  {"unit": "bar",   "hz": 100, "type": "pressure"},
    "PS6":  {"unit": "bar",   "hz": 100, "type": "pressure"},
    "EPS1": {"unit": "W",     "hz": 100, "type": "motor_power"},
    "FS1":  {"unit": "l/min", "hz": 10,  "type": "flow"},
    "FS2":  {"unit": "l/min", "hz": 10,  "type": "flow"},
    "TS1":  {"unit": "°C",    "hz": 1,   "type": "temperature"},
    "TS2":  {"unit": "°C",    "hz": 1,   "type": "temperature"},
    "TS3":  {"unit": "°C",    "hz": 1,   "type": "temperature"},
    "TS4":  {"unit": "°C",    "hz": 1,   "type": "temperature"},
    "VS1":  {"unit": "mm/s",  "hz": 1,   "type": "vibration"},
    "CE":   {"unit": "%",     "hz": 1,   "type": "cooling_efficiency"},
    "CP":   {"unit": "kW",    "hz": 1,   "type": "cooling_power"},
    "SE":   {"unit": "%",     "hz": 1,   "type": "efficiency"},
}

EXPECTED_COLS = {name: meta["hz"] * 60 for name, meta in SENSOR_META.items()}

# ── Label definitions (from UCI documentation) ─────────────────────
LABEL_COLS = ["cooler", "valve", "pump", "accumulator", "stable"]
TARGETS    = ["cooler", "valve", "pump", "accumulator"]

# Original integer value → human-readable string
LABEL_MAPS = {
    "cooler":      {3: "near failure",    20: "reduced efficiency", 100: "full efficiency"},
    "valve":       {73: "near failure",   80: "severe lag",         90: "small lag",        100: "optimal"},
    "pump":        {0: "no leakage",      1: "weak leakage",        2: "severe leakage"},
    "accumulator": {90: "near failure",   100: "severely reduced",  115: "slightly reduced", 130: "optimal"},
}

# Original value → ordinal class integer (0 = worst, highest = best)
LABEL_ENCODINGS = {
    "cooler":      {3: 0,  20: 1,  100: 2},
    "valve":       {73: 0, 80: 1,   90: 2,  100: 3},
    "pump":        {0: 0,  1: 1,    2: 2},
    "accumulator": {90: 0, 100: 1, 115: 2,  130: 3},
}

# Reverse: encoded int → description (used by the dashboard)
LABEL_DECODINGS = {
    target: {v: LABEL_MAPS[target][k] for k, v in enc.items()}
    for target, enc in LABEL_ENCODINGS.items()
}

print("Configuration loaded ✓")

## 2.1 — Load raw sensor files

In [ ]:
print(f"Loading from '{RAW_DIR}/'...\n")

sensors = {}

for name, meta in SENSOR_META.items():
    filepath = os.path.join(RAW_DIR, f"{name}.txt")
    df       = pd.read_csv(filepath, sep="\t", header=None)
    sensors[name] = df

    expected = EXPECTED_COLS[name]
    status   = "✓" if df.shape[1] == expected else f"⚠ expected {expected} cols"
    print(f"  {name:<6} {df.shape[0]} cycles × {df.shape[1]:>5} timesteps   "
          f"({meta['hz']}Hz, {meta['unit']})   {status}")

# Load profile (labels)
profile = pd.read_csv(
    os.path.join(RAW_DIR, "profile.txt"),
    sep="\t", header=None, names=LABEL_COLS
)
print(f"\n  {'profile':<6} {profile.shape[0]} cycles × {profile.shape[1]} label columns   ✓")

# All files must have the same number of cycles
all_counts = [df.shape[0] for df in sensors.values()] + [profile.shape[0]]
assert len(set(all_counts)) == 1, "Cycle count mismatch across files"

N_TOTAL = profile.shape[0]
print(f"\nAll {len(sensors)} sensor files + profile consistent: {N_TOTAL} cycles each ✓")

## 2.2 — Filter unstable cycles

The `stable` column flags cycles where the rig hadn't fully settled into its operating conditions. A value of `1` means the sensor readings may not reliably reflect the true component health state — so we drop those rows before training.

In [ ]:
stable_mask = profile["stable"] == 0
N_STABLE    = int(stable_mask.sum())
N_DROPPED   = N_TOTAL - N_STABLE

print(f"  Total cycles:    {N_TOTAL}")
print(f"  Stable  (keep):  {N_STABLE}  ({100 * N_STABLE / N_TOTAL:.1f}%)")
print(f"  Dropped (noise): {N_DROPPED} ({100 * N_DROPPED / N_TOTAL:.1f}%)")

# Apply the same boolean mask to every sensor array
sensors_clean = {
    name: df[stable_mask].reset_index(drop=True)
    for name, df in sensors.items()
}

print(f"\nAll {len(sensors_clean)} sensor arrays filtered to {N_STABLE} rows ✓")

## 2.3 — Extract and encode labels

In [ ]:
# Raw labels: drop the 'stable' flag, keep prediction targets only
labels_raw = (
    profile[stable_mask][TARGETS]
    .reset_index(drop=True)
    .copy()
)

# Encoded labels: map original values → 0-indexed ordinal integers
labels_encoded = labels_raw.copy()
for col, mapping in LABEL_ENCODINGS.items():
    labels_encoded[col] = labels_raw[col].map(mapping)

assert labels_encoded.isna().sum().sum() == 0, "Encoding produced NaN — unexpected label value"

print("Label encoding complete\n")
print("Raw labels (original UCI values) — first 5 rows:")
display(labels_raw.head())
print("\nEncoded labels (0 = worst, highest = best) — first 5 rows:")
display(labels_encoded.head())

In [ ]:
# Full class distribution
print("Class distributions after filtering:\n")
for col in TARGETS:
    counts = labels_raw[col].value_counts().sort_index()
    print(f"  {col.capitalize()}:")
    for val, count in counts.items():
        desc = LABEL_MAPS[col][val]
        pct  = 100 * count / N_STABLE
        bar  = "█" * int(pct / 2)
        print(f"    {val:>4}  ({desc:<22})  {count:>4} cycles  {pct:5.1f}%  {bar}")
    print()

## 2.4 — Data integrity validation

In [ ]:
print("Running integrity checks...\n")

for name, df in sensors_clean.items():
    assert df.shape[0] == N_STABLE,            f"{name}: row count mismatch"
    assert df.isna().sum().sum() == 0,         f"{name}: contains NaN"
    assert df.shape[1] == EXPECTED_COLS[name], f"{name}: wrong column count"

for col, enc in LABEL_ENCODINGS.items():
    expected_vals = set(enc.values())
    actual_vals   = set(labels_encoded[col].unique())
    assert actual_vals <= expected_vals, \
        f"{col}: unexpected encoded values {actual_vals - expected_vals}"

print(f"  ✓ All {len(sensors_clean)} sensor arrays — {N_STABLE} rows, no nulls, correct shapes")
print(f"  ✓ Encoded label values within expected ranges for all 4 targets")
print("\nAll checks passed — safe to save and proceed to feature engineering ✓")

## 2.5 — Save processed data

In [ ]:
# Sensor arrays → parquet (far faster to reload than CSV for wide data)
for name, df in sensors_clean.items():
    df.to_parquet(os.path.join(SENSORS_DIR, f"{name}.parquet"), index=False)

# Labels
labels_raw.to_csv(os.path.join(PROCESSED_DIR, "labels_raw.csv"), index=False)
labels_encoded.to_csv(os.path.join(PROCESSED_DIR, "labels_encoded.csv"), index=False)

# Mappings for dashboard display (convert int keys → str for JSON)
def int_keys(d):
    return {str(k): int_keys(v) for k, v in d.items()} if isinstance(d, dict) else d

with open(os.path.join(PROCESSED_DIR, "label_mappings.json"), "w") as f:
    json.dump(int_keys({
        "label_maps":      LABEL_MAPS,
        "label_encodings": LABEL_ENCODINGS,
        "label_decodings": LABEL_DECODINGS,
        "sensor_meta":     SENSOR_META,
    }), f, indent=2)

# Cleaning summary
with open(os.path.join(PROCESSED_DIR, "cleaning_report.json"), "w") as f:
    json.dump({
        "total_cycles":   N_TOTAL,
        "stable_cycles":  N_STABLE,
        "dropped_cycles": N_DROPPED,
        "stable_pct":     round(100 * N_STABLE / N_TOTAL, 1),
        "label_counts":   {
            col: {str(k): int(v)
                  for k, v in labels_raw[col].value_counts().items()}
            for col in TARGETS
        },
    }, f, indent=2)

print("Saved to data/processed/ :\n")
print(f"  sensors/             {len(sensors_clean)} .parquet files")
print(f"  labels_raw.csv       {labels_raw.shape}")
print(f"  labels_encoded.csv   {labels_encoded.shape}")
print(f"  label_mappings.json")
print(f"  cleaning_report.json")

## 2.6 — Diagnostic plots

In [ ]:
# ── Plot 1: Class distributions for all 4 targets ──────────────────
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
fig.patch.set_facecolor("#0d1117")
palette = ["#7F77DD", "#1D9E75", "#EF9F27", "#D85A30"]

for ax, col, color in zip(axes, TARGETS, palette):
    counts      = labels_raw[col].value_counts().sort_index()
    tick_labels = [LABEL_MAPS[col][v] for v in counts.index]
    bars = ax.bar(tick_labels, counts.values,
                  color=color, alpha=0.85, edgecolor="#1a1a2e", linewidth=0.5)
    ax.set_facecolor("#161b22")
    ax.set_title(col.capitalize(), color="white", fontsize=11, pad=8)
    ax.tick_params(colors="#888", labelsize=8)
    ax.set_xticklabels(tick_labels, rotation=30, ha="right")
    for spine in ax.spines.values():
        spine.set_edgecolor("#30363d")
    for bar in bars:
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 4,
                str(int(bar.get_height())), ha="center", color="white", fontsize=9)

fig.suptitle(f"Class distributions — {N_STABLE} stable cycles",
             color="white", fontsize=12, y=1.01)
plt.tight_layout()
plt.savefig(os.path.join(PROCESSED_DIR, "plot_class_distributions.png"),
            dpi=150, bbox_inches="tight", facecolor="#0d1117")
plt.show()

In [ ]:
# ── Plot 2: Raw time-series signal for one cycle ───────────────────
# Tweak these to inspect any cycle or sensor
CYCLE_IDX = 0
SENSOR    = "PS1"

signal    = sensors_clean[SENSOR].iloc[CYCLE_IDX].values
hz        = SENSOR_META[SENSOR]["hz"]
unit      = SENSOR_META[SENSOR]["unit"]
time_axis = np.arange(len(signal)) / hz
lbl_dict  = labels_raw.iloc[CYCLE_IDX].to_dict()
lbl_str   = "   |   ".join(f"{k}: {LABEL_MAPS[k][v]}" for k, v in lbl_dict.items())

fig, ax = plt.subplots(figsize=(13, 3))
fig.patch.set_facecolor("#0d1117")
ax.set_facecolor("#161b22")
ax.plot(time_axis, signal, color="#7F77DD", linewidth=0.7, alpha=0.9)
ax.set_xlabel("Time (s)", color="#888", fontsize=10)
ax.set_ylabel(f"{SENSOR} ({unit})", color="#888", fontsize=10)
ax.tick_params(colors="#888")
for spine in ax.spines.values():
    spine.set_edgecolor("#30363d")
ax.set_title(f"Cycle {CYCLE_IDX} — {SENSOR} raw signal ({hz}Hz, 60 s)\n{lbl_str}",
             color="white", fontsize=10, pad=8)
plt.tight_layout()
plt.show()

print(f"Stats — mean: {signal.mean():.3f}  std: {signal.std():.3f}  "
      f"min: {signal.min():.3f}  max: {signal.max():.3f}")

In [ ]:
# ── Imbalance check ────────────────────────────────────────────────
print("Class imbalance check (flag if < 80 samples):\n")
all_ok = True
for col in TARGETS:
    for val, count in labels_raw[col].value_counts().sort_index().items():
        desc = LABEL_MAPS[col][val]
        if count < 80:
            print(f"  ⚠  {col} '{desc}': only {count} — use stratified split")
            all_ok = False
        else:
            print(f"  ✓  {col} '{desc}': {count}")

print()
if all_ok:
    print("All classes have sufficient samples ✓")
else:
    print("Stratified split is essential for flagged classes ⚠")

## 2.7 — Summary

| | |
|---|---|
| Raw cycles loaded | 2205 |
| Stable cycles kept | see cell 2.2 output |
| Unstable cycles dropped | see cell 2.2 output |
| Sensors | 17 |
| Prediction targets | 4 (cooler, valve, pump, accumulator) |
| Saved to | `data/processed/` |

**Next → `03_feature_engineering.ipynb`**

Collapse each cycle's raw time-series (43,680 columns) into ~136 statistical features per cycle — mean, std, min, max, RMS, skewness, kurtosis per sensor — ready for the XGBoost classifier.